In [ ]:
# mount Google Drive
from google.colab import drive
drive.mount('/content/drive');

# import libraries
import scipy.sparse as sp
from scipy.sparse import coo_array
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textblob import TextBlob
import nltk
nltk.download('punkt');
nltk.download('stopwords');
from nltk.corpus import stopwords
from pprint import pprint
from collections import OrderedDict
import pickle
import math
from numba import none
from zmq import NULL
import string
from scipy.stats.distributions import chi2
from sklearn.metrics import confusion_matrix
import time
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet');
from nltk.tokenize import word_tokenize
import tensorflow as tf
import keras
!pip install keras_visualizer
import keras_visualizer
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold

MessageError: Error: credential propagation was unsuccessful

## ***Dataframe Cleaning***

Clean the raw dataframe from the previous subsection. This includes checking for and removing null instances, removing stop-words using a known list of common stop-words, and removing infrequently used terms. The frequency threshold can be adjusted by changing the value of thresh to a percentage of documents in the corpus. By default, it is 0.001, meaning that terms occurring in less than 150 documents are removed from the dictionary.

In [ ]:
# @title
# step 1: remove null values
print("Removed", df.isnull().any(axis=1).sum(), "null instances: ", df.shape[0], " --> ", end = "");
df = df.dropna();
print(df.shape[0], "\n");
n = df.shape[0];

# step 2: remove stopwords
print("Stopwords: ", stopwords.words('english'));
stop_words = set(stopwords.words('english'));
print("Searching for ", len(stop_words), " stop-words to remove...");

for word in stop_words:
  word = word.replace("'", "");
  df = df[df['term'] != word];

print("Removed", n - df.shape[0] ,"stop-words: ", n, " --> ", df.shape[0], "\n");
n = df.shape[0];

# step 3: reset DF indices and filter to words that have a frequency of less than 0.1% of the corpus.
ndocs = 150000;
thresh = ndocs * 0.001;
print("Removing instances that occur in less than", thresh, "documents.");

df.reset_index(inplace = True);
df = df.drop(['index'], axis = 1);
df_filtered = df[df['absDocFreq'] >= thresh];

# step 4: re-sort
df_filtered = df_filtered.sort_values(by = ['allAbsFreq'], ascending = False, ignore_index = True);
print("Removed", n - df_filtered.shape[0] ,"low-frequency instances: ", n, " --> ", df_filtered.shape[0]);

# step 5: compute IDF
df_filtered['logIDF'] = np.log(150000 / (df_filtered['absDocFreq'] + 1));
df_filtered['logIDF2c'] = np.log(150000 / (df_filtered['absDocFreq2c'] + 1));

In [ ]:
# print
df_filtered.to_csv('/content/drive/MyDrive/ECE_537_CIS_568/Data/Intermediate/tfDataFrameFiltered.csv', index = False);
df_filtered

# ***Decision Tree***

In [ ]:
# load and uncompress feature matrices for 3 classes
trainMatrix = sp.load_npz("/content/drive/MyDrive/ECE_537_CIS_568/Data/Intermediate/trainSparseFeatureMatrixChiSq3c250.npz");
trainMatrix = trainMatrix.todense();
testMatrix = sp.load_npz("/content/drive/MyDrive/ECE_537_CIS_568/Data/Intermediate/testSparseFeatureMatrixChiSq3c250.npz");
testMatrix = testMatrix.todense();

trainArray3c = np.array(trainMatrix)
testArray3c = np.array(testMatrix)


# create model parameters
ytrn3c = trainArray3c[:,0] # should I use copy?
xtrn3c = trainArray3c[:,1].reshape(-1,1)

ytest3c = testArray3c[:,0]
xtest3c = testArray3c[:,1].reshape(-1,1)

# # importing the decision tree classifier from sklearn
# from sklearn.tree import DecisionTreeClassifier

# # classification report and confusion matrix
# from sklearn.metrics import classification_report,confusion_matrix,accuracy_score


# # generating a confusion matrix
# from sklearn import metrics
# from sklearn.model_selection import StratifiedKFold

# intializing decision tree classfiier
dtc = DecisionTreeClassifier()

# # using grid search to test over given hyperparamer values
# from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [5, 8, 10, 12, 14, 16],
    'min_samples_split': [50, 100, 200, 300, 500, 1000],
    'min_samples_leaf': [50, 100, 200, 300, 500],
}


# Create StratifiedKFold
stratified_cv3c = StratifiedKFold(n_splits=5, shuffle=True, random_state=23)

# Create the GridSearchCV object with stratified sampling
grid_search = GridSearchCV(dtc, param_grid, cv=stratified_cv3c, scoring='accuracy')


grid_search = GridSearchCV(dtc, param_grid, cv=10, scoring='accuracy')
grid_search.fit(xtrn3c, ytrn3c)

best_params = grid_search.best_params_
best_dt_model = grid_search.best_estimator_

# print("Best Parameters: ",best_params)
print("Best Model: ", best_dt_model)

# predicting on xtest
yproba3c = best_dt_model.predict_proba(xtest3c)[:,1]

ypred3c = best_dt_model.predict(xtest3c)

df_accuracy = accuracy_score(ytest3c, ypred3c)
print(f"Accuracy of Decison Tree Model: {df_accuracy}")

In [ ]:
# # generating a confusion matrix
# from sklearn import metrics

# cm3c = metrics.confusion_matrix(ytest3c, ypred3c)

# dtc_cm3c = pd.DataFrame(data=cm3c, columns=['predict: Good','predict: Bad', 'predict: Neutral'],
#                     index=['true: Good', 'true: Bad', 'true: Neutral'])
# classifier report
print(classification_report(ytest3c,ypred3c))



In [ ]:
# creating confusion matrix visualization
# import seaborn as sns
cm3c = metrics.confusion_matrix(ytest3c, ypred3c)

# dtc_cm3c = pd.DataFrame(data=cm3c, columns=['predict: Good','predict: Bad', 'predict: Neutral'],
#                     index=['true: Good', 'true: Bad', 'true: Neutral'])

sns.heatmap(cm3c,
            annot=True,
            fmt='g',
            linewidth=.5,
            xticklabels=['Pred: Good','Pred: Bad', 'Pred: Neutral'],
            yticklabels=['True: Good','True: Bad', 'True: Neutral'],
            cmap='viridis')

plt.title('Decision Tree 3-Class Confusion Matrix',fontsize=17)
plt.show()

In [ ]:
# visualizing the actual decision tree
# from sklearn import tree

feature_names = trainData.columns

dtc_fig = plt.figure(figsize=(20,10))

_ = tree.plot_tree(best_dt_model,
                  feature_names=feature_names,
                  class_names={0:'Good', 1: 'Neutral', 2: 'Bad'},
                  filled=True,
                  fontsize=8)



# ***Decision Tree - 2 Classifier***

In [ ]:
# intializing decision tree classfiier
dtc = DecisionTreeClassifier()

# using grid search to test over given hyperparamer values
# from sklearn.model_selection import GridSearchCV

param_grid2c = {
    'max_depth': [5, 8, 10, 12, 14, 16],
    'min_samples_split': [50, 100, 200, 300, 500, 1000],
    'min_samples_leaf': [50, 100, 200, 300, 500],
}

# Create StratifiedKFold
stratified_cv2c = StratifiedKFold(n_splits=5, shuffle=True, random_state=33)

# Create the GridSearchCV object with stratified sampling
grid_search = GridSearchCV(dtc, param_grid2c, cv=stratified_cv2c, scoring='accuracy')

grid_search.fit(xtrn2c, ytrn2c)

best_params = grid_search.best_params_
best_dt_model2c = grid_search.best_estimator_

# print("Best Parameters: ",best_params)
print("Best Model: ", best_dt_model2c)

# predicting on xtest
yproba = best_dt_model2c.predict_proba(xtest2c)[:,1]

ypred2c = best_dt_model2c.predict(xtest2c)



In [ ]:
# classifier report
print(classification_report(ytest2c, ypred2c))

# generating a confusion matrix
# from sklearn import metrics

cm2c = metrics.confusion_matrix(ytest2c, ypred2c)

sns.heatmap(cm2c,
            annot=True,
            fmt='g',
            linewidth=.5,
            xticklabels=['Pred: Good', 'Pred: Bad'],
            yticklabels=['True: Good','True: Bad'],
            cmap='viridis')

plt.title('Decison Tree 2-Class Confusion Matrix',fontsize=17)
plt.show()

In [ ]:
# visualizing the actual decision tree
# from sklearn import tree

feature_names = trainData.columns

dtc_fig = plt.figure(figsize=(20,10))

figure = tree.plot_tree(best_dt_model2c,
                  feature_names=feature_names,
                  class_names={0:'Good', 1: 'Bad'},
                  filled=True,
                  fontsize=8)


df_accuracy = accuracy_score(ytest2c, ypred2c)
print(f"Accuracy of Decison Tree Model: {df_accuracy}")

# ***K-NN Model***

In [ ]:
# load and uncompress feature matrices for 3 classes
trainMatrix = sp.load_npz("/content/drive/MyDrive/ECE_537_CIS_568/Data/Intermediate/trainSparseFeatureMatrixChiSq3c250.npz");
trainMatrix = trainMatrix.todense();
testMatrix = sp.load_npz("/content/drive/MyDrive/ECE_537_CIS_568/Data/Intermediate/testSparseFeatureMatrixChiSq3c250.npz");
testMatrix = testMatrix.todense();

trainArray3c = np.array(trainMatrix)
testArray3c = np.array(testMatrix)


# create model parameters
ytrn3c = trainArray3c[:,0] # should I use copy?
xtrn3c = trainArray3c[:,1].reshape(-1,1)

ytest3c = testArray3c[:,0]
xtest3c = testArray3c[:,1].reshape(-1,1)


In [ ]:
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.metrics import accuracy_score


knn_clf=KNeighborsClassifier()


# Create StratifiedKFold
stratified_KNN3c = StratifiedKFold(n_splits=5, shuffle=True, random_state=57)

# defining a range of values for n_neighbors to test in grid search
knn_param_grid3c = {'n_neighbors' : [25, 50, 75, 100, 125],
              'metric' : ['euclidean', 'manhattan']}

# create a grid search to assess best amount of n_neighbors per cluster
knn_grid_search3c = GridSearchCV(knn_clf, knn_param_grid3c, cv=stratified_KNN3c)

# fit the knn model to the training data
knn_grid_search3c.fit(xtrn3c, ytrn3c)

# printing the best hyperparameters found by the grid search
print("Best Hyperparameters:", knn_grid_search3c.best_params_)

# identifying the best model
best_knn_model3c = knn_grid_search3c.best_estimator_
print('Best KNN Model: ', best_knn_model3c)

# make predictions on the test data
knn_preds3c = best_knn_model3c.predict(xtest3c)




In [ ]:
# printing exact accuracy score
knn_accuracy3c = accuracy_score(ytest3c, knn_preds3c)
print(f"Accuracy of Decison Tree Model: {knn_accuracy3c}")

# evaluating accruacy with classfication report
print(classification_report(ytest3c,knn_preds3c))

# generating a confusion matrix
# from sklearn import metrics

Knn_cm3c = metrics.confusion_matrix(ytest3c, ypred3c)

sns.heatmap(Knn_cm3c,
            annot=True,
            fmt='g',
            linewidth=.5,
            xticklabels=['Pred: Good','Pred: Bad', 'Pred: Neutral'],
            yticklabels=['True: Good','True: Bad', 'True: Neutral'],
            cmap='viridis')

plt.title('KNN 3-Class Confusion Matrix',fontsize=17)
plt.show()

# KNN 2-Class Classifier

In [ ]:
# load and uncompress feature matrices for 3 classes
trainMatrix = sp.load_npz("/content/drive/MyDrive/ECE_537_CIS_568/Data/Intermediate/trainSparseFeatureMatrixChiSq2c250.npz");
trainMatrix = trainMatrix.todense();
testMatrix = sp.load_npz("/content/drive/MyDrive/ECE_537_CIS_568/Data/Intermediate/testSparseFeatureMatrixChiSq2c250.npz");
testMatrix = testMatrix.todense();

trainArray2c = np.array(trainMatrix)
testArray2c = np.array(testMatrix)


# create model parameters
ytrn2c = trainArray2c[:,0] # should I use copy?
xtrn2c = trainArray2c[:,1].reshape(-1,1)

ytest2c = testArray2c[:,0]
xtest2c = testArray2c[:,1].reshape(-1,1)

In [ ]:
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.metrics import accuracy_score


knn_clf=KNeighborsClassifier()

# Create StratifiedKFold
stratified_KNN2c = StratifiedKFold(n_splits=5, shuffle=True, random_state=66)

# defining a range of values for n_neighbors to test in grid search
knn_param_grid2c = {'n_neighbors' : [50, 75, 100, 125, 150, 200],
              'metric' : ['euclidean', 'manhattan']}

# create a grid search to assess best amount of n_neighbors per cluster
knn_grid_search2c = GridSearchCV(knn_clf, knn_param_grid2c, cv=stratified_KNN2c)  # 5-fold cross-validation

# fit the knn model to the training data
knn_grid_search2c.fit(xtrn2c, ytrn2c)

# printing the best hyperparameters found by the grid search
print("Best Hyperparameters:", knn_grid_search2c.best_params_)

# identifying the best model
best_knn_model2c = knn_grid_search2c.best_estimator_
# print('Best KNN Model: ', best_knn_model)

# make predictions on the test data
knn_preds2c = best_knn_model2c.predict(xtest2c)




In [ ]:
# printing exact accuracy score
knn_accuracy2c = accuracy_score(ytest2c, knn_preds2c)
print(f"Accuracy of Decison Tree Model: {knn_accuracy2c}")

# evaluating accruacy with classfication report
print(classification_report(ytest2c, knn_preds2c))

# generating a confusion matrix
# from sklearn import metrics

Knn_cm2c = metrics.confusion_matrix(ytest2c, knn_preds2c)

sns.heatmap(Knn_cm2c,
            annot=True,
            fmt='g',
            linewidth=.5,
            xticklabels=['Pred: Good', 'Pred: Bad'],
            yticklabels=['True: Good', 'True: Bad'],
            cmap='viridis')

plt.title('KNN 2-Class Confusion Matrix',fontsize=17)
plt.show()